Install dependencies and load data

In [1]:
!pip install transformers datasets scikit-learn torch accelerate tqdm -q

In [2]:
import os, random, numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from transformers import (
    RobertaTokenizerFast, RobertaForSequenceClassification,
)
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DIR   = './roberta_classifier_final'
INFER_BATCH = 128
MAX_LEN      = 512
MODEL_NAME   = 'roberta-base'

LABEL2ID = {'not_green_claim': 0, 'green_claim': 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

c:\Users\willi\CPSC449\canada-oil-greenwash-scraping\greenwash-scraping\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("labelled.csv")
df = df[df["Label"] != -1]
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

unlabelled_df = pd.read_csv("unlabelled.csv")

# For duplicate sentences in unlabelled, give them the same label as the human labelling
label_map = dict(zip(df["Sentence"], df["Label"]))
unlabelled_df["Label"] = unlabelled_df["Sentence"].map(label_map).fillna(-1)

Label with pre-trained model

In [4]:
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

best_threshold = float(np.load(os.path.join(MODEL_DIR, "best_threshold.npy"))[0])

best_model = RobertaForSequenceClassification.from_pretrained(
    os.path.abspath(MODEL_DIR)
).to(DEVICE)
best_model.eval()

to_label = unlabelled_df["Label"] == -1
sentences = unlabelled_df.loc[to_label, "Sentence"].tolist()

all_binary_labels = []

for i in tqdm(range(0, len(sentences), INFER_BATCH), desc="Labelling"):
    batch_texts = sentences[i : i + INFER_BATCH]
    enc = tokenizer(
        batch_texts,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        probs = torch.softmax(best_model(**enc).logits, dim=-1)[:, 1].cpu().numpy()

    labels = (probs > best_threshold).astype(int)
    all_binary_labels.extend(labels.tolist())

unlabelled_df.loc[to_label, "Label"] = all_binary_labels

print(f"Total labelled: {len(sentences)}")
print(f"Green claims: {(sum(all_binary_labels))}")
print(f"Not green claims: {(len(all_binary_labels) - sum(all_binary_labels))}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Labelling:   0%|          | 0/494 [00:00<?, ?it/s]

Total labelled: 63170
Green claims: 4285
Not green claims: 58885


In [5]:
combined_df = pd.concat([df, unlabelled_df], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
combined_df.to_csv("all_labelled.csv", index=False)

In [3]:
combined_df = pd.read_csv("../../output/analyzed/all_labelled.csv")

Basic dataset summary statistics

In [4]:
_PERIOD = {True: "pre-legislation", False: "post-legislation"}
_METRIC = {
    "green_claims": "Green Claims",
    "total_rows": "Total Sentences",
    "normalized": "Normalized",
}

_COL_ORDER = [f"{m} ({p})" for m in _METRIC.values() for p in _PERIOD.values()]

def summary_table(df):
    return df.agg(
        **{"Total Sentences": ("Label", "count"), "Green Claims": ("Label", "sum")}
    ).assign(Normalized=lambda x: (x["Green Claims"] / x["Total Sentences"]).round(4))


wayback_table = summary_table(combined_df.groupby("isWayback"))
print("=== Before vs. After Bill C-59 ===")
print(wayback_table.to_string())

org_table = summary_table(combined_df.groupby("Organization")).sort_values(
    "Green Claims", ascending=False
)
print("\n=== By Organization ===")
print(org_table.to_string())

org_wayback_pivot = (
    combined_df.groupby(["Organization", "isWayback"])
    .agg(green_claims=("Label", "sum"), total_rows=("Label", "count"))
    .assign(normalized=lambda x: (x["green_claims"] / x["total_rows"]).round(4))
    .unstack("isWayback")
)

org_wayback_pivot.columns = [
    f"{_METRIC[m]} ({_PERIOD[wb]})" for m, wb in org_wayback_pivot.columns
]
org_wayback_pivot = org_wayback_pivot[_COL_ORDER]

print("\n=== Green Claims by Organization (pre-legislation vs post-legislation) ===")
print(org_wayback_pivot.to_string())

=== Before vs. After Bill C-59 ===
           Total Sentences  Green Claims  Normalized
isWayback                                           
False                23127          1680      0.0726
True                 44749          4141      0.0925

=== By Organization ===
                            Total Sentences  Green Claims  Normalized
Organization                                                         
Enbridge                              12214          1660      0.1359
Suncor Energy                         15346          1401      0.0913
Pembina Pipeline                      25341          1143      0.0451
Canadian Natural Resources             7691           575      0.0748
Shell Canada                           2275           573      0.2519
Imperial Oil                           5009           469      0.0936

=== Green Claims by Organization (pre-legislation vs post-legislation) ===
                            Green Claims (pre-legislation)  Green Claims (post-legislation) 